In [4]:
from pyspark.sql import SparkSession

spark=SparkSession\
.builder \
.appName("read from socket") \
.master("local[*]") \
.getOrCreate()

spark

In [8]:
#Read Input
df_raw=spark.read.format("text").load("example.txt")
df_raw.printSchema()

root
 |-- value: string (nullable = true)



In [13]:
#
from pyspark.sql.functions import split

df_splt=df_raw.withColumn("words",split("value"," "))
df_splt.show()

+--------------------+--------------------+
|               value|               words|
+--------------------+--------------------+
|a quick brown fox...|[a, quick, brown,...|
+--------------------+--------------------+



In [16]:
#explode
from pyspark.sql.functions import explode

df_explode=df_splt.withColumn("word",explode("words")).drop("value","words")
df_explode.show()

+-----+
| word|
+-----+
|    a|
|quick|
|brown|
|  fox|
|jumps|
| over|
|    a|
| lazy|
|  dog|
+-----+



In [20]:
from pyspark.sql.functions import lit,count
df_agg=df_explode.groupBy("word").agg(count(lit (1)).alias("CNT"))
df_agg.show()

+-----+---+
| word|CNT|
+-----+---+
| lazy|  1|
|jumps|  1|
|  dog|  1|
|  fox|  1|
|brown|  1|
| over|  1|
|quick|  1|
|    a|  2|
+-----+---+

